# Vision Transformer（ViT）從零開始：CIFAR-10 RGB 圖像分類
### *An Image is Worth 16×16 Words* — a from-scratch, training-first tour

本筆記本逐層手刻論文 **"An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale"** (Dosovitskiy et al., ICLR 2021) 的 Vision Transformer，並在 **RGB 的 CIFAR-10** 上**從零開始訓練**。重點放在**模型結構**——每個張量怎麼流動、每個設計選擇為何如此。

> **這是理解 DINO / cellDINO 的基石。** 本筆記本只談「基本 ViT」（忠於原論文）；DINO 的自監督部分留到另一本筆記本。我們在這裡刻意採用論文原版的 `[CLS]` token 與可學習位置編碼，因為那正是 DINO backbone 的形狀。

#### 設計決策（design tree，已定案）
| 決策 | 選擇 | 理由 |
|---|---|---|
| Dataset | CIFAR-10 (32×32 RGB) | 從零訓練 ViT 的標準教學資料集；patch 4 → 64 tokens |
| 架構 | 論文原版：`[CLS]` + 可學習 1D 位置編碼 + Pre-LN + GELU | 即 DINO backbone 形狀 |
| 規模 | Tiny：dim 192 / depth 6 / heads 3 / MLP×4（~2–3M） | 在 Mac MPS 上「看得到它學起來」 |
| 訓練 | AdamW + warmup→cosine、RandAugment + crop/flip + label smoothing | ViT 無歸納偏好，靠資料增強補足 |
| 分析 | 結構+shape trace · patch/濾波器 PCA · attention rollout · 位置編碼/注意力距離 | 1:1 對應論文 Fig 6 & 7 |

#### ⚠️ 論文的核心教訓（§4.3, Fig 3/4）
ViT **缺乏 CNN 的歸納偏好**（局部性 locality、平移等變性 translation equivariance）。因此**在小資料上從零訓練會輸給同規模的 CNN**。我們用強資料增強去逼近一個堪用的模型（~75–80%），但請把這個「資料飢渴」現象本身當成一個重要觀察——它正是後續需要大規模（自）監督預訓練（如 DINO）的動機。

## 1. 環境與裝置 / Setup & device
我們用 PyTorch + MPS（Apple Silicon GPU）。`SMOKE` 旗標供自動化煙霧測試使用（少量資料 + 1 epoch）；正常使用時忽略即可。

In [ ]:
# %pip install einops vit-pytorch scikit-learn --quiet   # 若環境缺套件，取消註解執行一次
import os, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)

SMOKE = os.environ.get("VIT_SMOKE", "0") == "1"   # 煙霧測試：少量資料 + 1 epoch

def get_device():
    if torch.backends.mps.is_available(): return torch.device("mps")
    if torch.cuda.is_available():         return torch.device("cuda")
    return torch.device("cpu")

DEVICE = get_device()
print("Device:", DEVICE, "| SMOKE:", SMOKE)

## 2. 超參數設定 / Configuration
所有結構與訓練超參數集中在這裡，方便你調整。

序列長度 = `(image/patch)² + 1`。CIFAR 是 32×32，patch=4 → 8×8 = **64 個 patch token**，再加 1 個 `[CLS]` → 序列長度 **65**。注意力成本是 **O(序列長度²)**，所以 patch size 是訓練速度的最大旋鈕。

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    image_size: int = 32
    patch_size: int = 4
    in_channels: int = 3
    num_classes: int = 10
    # ── 模型結構 ──
    dim: int = 192          # D：Transformer 隱藏維度（latent size），整條網路保持不變
    depth: int = 6          # L：Encoder block 層數
    heads: int = 3          # 注意力頭數（head_dim = dim/heads = 64）
    mlp_ratio: int = 4      # MLP 隱藏層 = dim * mlp_ratio
    dropout: float = 0.1
    # ── 訓練 ──
    batch_size: int = 128
    epochs: int = 50
    lr: float = 3e-4
    weight_decay: float = 0.05
    warmup_epochs: int = 5
    label_smoothing: float = 0.1

cfg = Config()
if SMOKE:
    cfg.epochs = 1; cfg.warmup_epochs = 0

cfg.num_patches = (cfg.image_size // cfg.patch_size) ** 2
cfg.head_dim    = cfg.dim // cfg.heads
cfg.grid        = cfg.image_size // cfg.patch_size
print(cfg)
print(f"num_patches={cfg.num_patches} | seq_len(+CLS)={cfg.num_patches+1} | head_dim={cfg.head_dim} | grid={cfg.grid}x{cfg.grid}")

## 3. 資料 / CIFAR-10 with augmentation
訓練增強：`RandomCrop(padding=4)` + `RandomHorizontalFlip` + `RandAugment` + 標準化。
測試只做標準化。**RandAugment 必須作用在 `ToTensor` 之前**（它吃 PIL 影像）。

> 為什麼這麼重視增強？因為 ViT 沒有 CNN 的局部性先驗，50k 張小圖很容易被它「背起來」。增強是把多樣性硬塞進資料分布，逼模型學到可泛化的模式。

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_set = torchvision.datasets.CIFAR10(root="./data", train=True,  download=True, transform=train_tf)
test_set  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_tf)
if SMOKE:
    train_set = Subset(train_set, range(1024))
    test_set  = Subset(test_set,  range(512))

CLASSES = ("plane","car","bird","cat","deer","dog","frog","horse","ship","truck")
nw = 0 if SMOKE else 2
train_loader = DataLoader(train_set, batch_size=cfg.batch_size, shuffle=True,  num_workers=nw, drop_last=True)
test_loader  = DataLoader(test_set,  batch_size=cfg.batch_size, shuffle=False, num_workers=nw)
print("train batches:", len(train_loader), "| test batches:", len(test_loader))

### 3.1 看一批增強後的樣本

In [ ]:
def denorm(t):
    # (C,H,W) 標準化張量 -> (H,W,C) in [0,1]，供顯示
    mean = torch.tensor(CIFAR_MEAN).view(3,1,1); std = torch.tensor(CIFAR_STD).view(3,1,1)
    return (t.cpu()*std + mean).clamp(0,1).permute(1,2,0).numpy()

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(14,4))
for ax, img, lb in zip(axes.flatten(), imgs, labels):
    ax.imshow(denorm(img)); ax.set_title(CLASSES[lb], fontsize=9); ax.axis("off")
plt.suptitle("CIFAR-10 樣本（已增強）/ augmented samples"); plt.tight_layout(); plt.show()

## 4. 把圖切成 patch / Patchify
ViT 的第一步：把 $\mathbf{x}\in\mathbb{R}^{H\times W\times C}$ 切成 $N=HW/P^2$ 個 $P\times P$ patch，攤平成序列。下圖把一張 32×32 圖切成 8×8 = 64 個 4×4 patch——這 64 個 patch 之後就是 Transformer 的「token」（等同 NLP 的「word」）。

In [ ]:
raw = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transforms.ToTensor())
img0, lb0 = raw[7]
P, G = cfg.patch_size, cfg.grid
fig, axes = plt.subplots(G, G, figsize=(5,5))
for i in range(G):
    for j in range(G):
        patch = img0[:, i*P:(i+1)*P, j*P:(j+1)*P]
        axes[i,j].imshow(patch.permute(1,2,0).numpy()); axes[i,j].axis("off")
plt.suptitle(f"一張 {cfg.image_size}x{cfg.image_size} 圖 -> {G}x{G}={cfg.num_patches} 個 {P}x{P} patches  (class={CLASSES[lb0]})")
plt.tight_layout(); plt.show()

## 5. Patch Embedding（Eq. 1 的線性投影）
論文用一個可訓練線性投影 $\mathbf{E}\in\mathbb{R}^{(P^2\cdot C)\times D}$ 把每個攤平的 patch 映到 $D$ 維。

**實作技巧**：`kernel=stride=P` 的 `Conv2d` 與「攤平 patch 再做 Linear」**數學等價**——非重疊卷積就是每個 patch 各做一次線性投影。這也是 DINO/timm 的標準寫法。

$$\text{(B, 3, 32, 32)} \xrightarrow{\text{Conv2d(k=s=4)}} \text{(B, 192, 8, 8)} \xrightarrow{\text{flatten+transpose}} \text{(B, 64, 192)}$$

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, in_channels, dim):
        super().__init__()
        self.num_patches = (image_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):           # x: (B, C, H, W)
        x = self.proj(x)            # (B, D, H/P, W/P)
        x = x.flatten(2)            # (B, D, num_patches)
        x = x.transpose(1, 2)       # (B, num_patches, D)
        return x

pe = PatchEmbedding(cfg.image_size, cfg.patch_size, cfg.in_channels, cfg.dim)
print("PatchEmbedding 輸出:", tuple(pe(torch.randn(2,3,32,32)).shape), " # (B, num_patches, D)")

## 6. Multi-Head Self-Attention（注意力的數學核心）
每個 token 投影出 query / key / value，注意力是縮放點積後的 softmax 加權：
$$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$
- **多頭**：把 $D$ 切成 `heads` 份（每份 `head_dim=64`），各自獨立做注意力再拼回，讓模型同時關注不同子空間。
- **$\sqrt{d_k}$ 縮放**：避免點積隨維度變大而把 softmax 推到飽和區、梯度消失。
- **全域性**：每個 token 都能看到所有 token——這就是 ViT 「沒有局部性先驗」的來源（與 CNN 的局部感受野相反）。

我們加一個 `store_attn` 旗標，需要時保存注意力權重，供後面 rollout / 注意力距離分析使用。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, dim, heads, dropout=0.0):
        super().__init__()
        assert dim % heads == 0
        self.heads, self.head_dim = heads, dim // heads
        self.scale = self.head_dim ** -0.5
        self.qkv  = nn.Linear(dim, dim * 3)     # 一次算出 Q,K,V
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)
        self.store_attn = False
        self.last_attn = None
    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.heads, self.head_dim).permute(2,0,3,1,4)
        q, k, v = qkv[0], qkv[1], qkv[2]                 # 各為 (B, heads, N, head_dim)
        attn = (q @ k.transpose(-2,-1)) * self.scale     # (B, heads, N, N)
        attn = attn.softmax(dim=-1)
        if self.store_attn: self.last_attn = attn.detach()
        out = (self.drop(attn) @ v)                      # (B, heads, N, head_dim)
        out = out.transpose(1,2).reshape(B, N, D)        # 拼回各 head
        return self.proj(out)

## 7. Feed-Forward（MLP block）
每個 token 獨立通過兩層 MLP（先升維 ×4 再降回），中間是 **GELU**（論文指定）。注意力負責 token 之間「混資訊」，MLP 負責每個 token 內部「做非線性變換」。

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, dim, mlp_ratio, dropout=0.0):
        super().__init__()
        hidden = dim * mlp_ratio
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, dim), nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

## 8. Transformer Encoder Block（Pre-LN + 殘差）
論文用 **Pre-LN**：LayerNorm 放在子層**之前**，殘差連接在之後（Eq. 2, 3）：
$$\mathbf{z}'_\ell=\text{MSA}(\text{LN}(\mathbf{z}_{\ell-1}))+\mathbf{z}_{\ell-1},\qquad
\mathbf{z}_\ell=\text{MLP}(\text{LN}(\mathbf{z}'_\ell))+\mathbf{z}'_\ell$$
Pre-LN 讓殘差路徑保持「乾淨」的恆等映射，深層 Transformer 才能穩定訓練（這也是現代 ViT/LLM 的標準做法，與原始 Transformer 的 Post-LN 不同）。

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, dim, heads, mlp_ratio, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim); self.attn = MultiHeadAttention(dim, heads, dropout)
        self.norm2 = nn.LayerNorm(dim); self.mlp  = FeedForward(dim, mlp_ratio, dropout)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))   # z'_l = MSA(LN(z_{l-1})) + z_{l-1}
        x = x + self.mlp(self.norm2(x))    # z_l  = MLP(LN(z'_l))   + z'_l
        return x

## 9. 組裝完整 ViT
對照論文 Eq. 1–4：
1. **Eq.1** patch 投影 + 前置 `[CLS]` token + 加位置編碼 → $\mathbf{z}_0$
2. **Eq.2–3** $L$ 層 encoder block
3. **Eq.4** 取 `[CLS]` 的輸出做 LayerNorm → 影像表徵 $\mathbf{y}$ → 線性分類頭

關鍵設計：
- **`[CLS]` token**：一個可學習向量，前置到序列最前。它本身不對應任何 patch，但透過注意力「彙整」全圖資訊，最終狀態當作整張圖的表徵（承自 BERT）。*DINO 著名的注意力圖正是來自這個 token。*
- **可學習 1D 位置編碼**：patch 攤平後失去 2D 位置資訊，加上 `pos_embed` 補回。論文發現可學習 1D 編碼已足夠，模型會自己學出 2D 拓樸（之後會驗證，Fig.7 center）。

In [ ]:
class ViT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.patch_embed = PatchEmbedding(cfg.image_size, cfg.patch_size, cfg.in_channels, cfg.dim)
        n = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, cfg.dim))        # 可學習 [CLS]
        self.pos_embed = nn.Parameter(torch.zeros(1, n + 1, cfg.dim))    # 可學習 1D 位置編碼 E_pos
        self.dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(cfg.dim, cfg.heads, cfg.mlp_ratio, cfg.dropout)
            for _ in range(cfg.depth)])
        self.norm = nn.LayerNorm(cfg.dim)
        self.head = nn.Linear(cfg.dim, cfg.num_classes)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
    def set_store_attn(self, flag):
        for blk in self.blocks: blk.attn.store_attn = flag
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)                       # (B, N, D)
        cls = self.cls_token.expand(B, -1, -1)        # (B, 1, D)
        x = torch.cat([cls, x], dim=1) + self.pos_embed   # z_0: (B, N+1, D)
        x = self.dropout(x)
        for blk in self.blocks: x = blk(x)
        x = self.norm(x)
        return self.head(x[:, 0])                     # 取 [CLS] -> 分類頭

model = ViT(cfg).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"ViT 參數量 / params: {n_params/1e6:.2f} M")

## 10. 形狀追蹤 / Tensor-shape trace
把一個 dummy batch 推過去，印出**每個階段的張量形狀**。這張流程圖（`(B,3,32,32) → (B,65,192) → (B,192) → (B,10)`）就是你之後讀 DINO 程式碼時要複用的心智模型。

In [ ]:
@torch.no_grad()
def trace_shapes(model, cfg):
    model.eval()
    x = torch.randn(2, cfg.in_channels, cfg.image_size, cfg.image_size, device=DEVICE)
    print(f"輸入 input              : {tuple(x.shape)}")
    p = model.patch_embed(x);                          print(f"patch embedding         : {tuple(p.shape)}   # (B, N, D)")
    cls = model.cls_token.expand(2, -1, -1)
    z = torch.cat([cls, p], 1) + model.pos_embed;      print(f"+CLS +pos_embed (z_0)   : {tuple(z.shape)}   # (B, N+1, D)")
    for blk in model.blocks: z = blk(z)
    z = model.norm(z);                                 print(f"after {cfg.depth} encoder blocks  : {tuple(z.shape)}")
    print(f"CLS token (y)           : {tuple(z[:,0].shape)}      # (B, D)")
    print(f"logits                  : {tuple(model.head(z[:,0]).shape)}       # (B, num_classes)")

trace_shapes(model, cfg)

### 10.1 結構總覽圖 / architecture diagram

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.6)); ax.axis("off")
def box(x, w, text, color, y=0.42, h=0.22):
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=color, edgecolor="black", lw=1.2))
    ax.text(x+w/2, y+h/2, text, ha="center", va="center", fontsize=9)
box(0.00, 0.13, "Image\n3×32×32", "#cfe8ff")
box(0.16, 0.14, "Patchify\n4×4 → 64", "#cfe8ff")
box(0.34, 0.15, "Linear proj\n→ 64×192", "#ffe6b3")
box(0.53, 0.13, "+[CLS]\n65×192", "#ffe6b3")
box(0.70, 0.12, "+Pos emb", "#ffe6b3")
box(0.86, 0.14, f"Encoder ×{cfg.depth}", "#d6f5d6", y=0.34, h=0.38)
for x0, x1 in [(0.13,0.16),(0.30,0.34),(0.49,0.53),(0.66,0.70),(0.82,0.86)]:
    ax.annotate("", xy=(x1,0.53), xytext=(x0,0.53), arrowprops=dict(arrowstyle="->"))
ax.text(0.93, 0.80, "[CLS] → LayerNorm → Linear → 10 類", ha="center", fontsize=9)
ax.set_xlim(0,1.05); ax.set_ylim(0.25, 0.9)
plt.title("ViT 結構總覽 / architecture overview"); plt.show()

## 11. 訓練設定 / Training setup
- **Optimizer**：AdamW（論文用 Adam + 高 weight decay；AdamW 是其現代等價版，把 weight decay 與梯度更新解耦）。
- **學習率排程**：線性 **warmup** → **cosine decay**（論文 App. B.1；也是 DeiT/DINO 的標準配方）。warmup 讓初期不穩的注意力先「暖機」。
- **Loss**：CrossEntropy + **label smoothing 0.1**（軟化標籤，抑制過度自信、改善泛化）。

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

steps_per_epoch = len(train_loader)
total_steps  = cfg.epochs * steps_per_epoch
warmup_steps = cfg.warmup_epochs * steps_per_epoch

def lr_lambda(step):
    if step < warmup_steps:                    # 線性 warmup
        return (step + 1) / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))   # cosine decay

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
print(f"steps/epoch={steps_per_epoch} | total={total_steps} | warmup={warmup_steps}")

### 11.1 訓練迴圈 / training loop
> 想跑完整訓練：把 `cfg.epochs` 調到 50（預設）。想快速看流程：改小即可。訓練完成的權重會存成 `vit_cifar10.pt`，後面的分析會用到訓練後的模型。

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct = total = 0; loss_sum = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        loss_sum += criterion(out, y).item() * x.size(0)
        correct  += (out.argmax(1) == y).sum().item(); total += x.size(0)
    return loss_sum/total, correct/total

history = {"train_loss":[], "test_loss":[], "test_acc":[], "lr":[]}

def train():
    for epoch in range(cfg.epochs):
        model.train(); t0 = time.time(); run_loss = total = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward(); optimizer.step(); scheduler.step()
            run_loss += loss.item()*x.size(0); total += x.size(0)
        te_loss, te_acc = evaluate(model, test_loader)   # 每個 epoch 在測試集評估
        tr_loss = run_loss/total
        history["train_loss"].append(tr_loss)
        history["test_loss"].append(te_loss); history["test_acc"].append(te_acc)
        history["lr"].append(scheduler.get_last_lr()[0])
        print(f"epoch {epoch+1:3d}/{cfg.epochs} | lr {history['lr'][-1]:.2e} | "
              f"train_loss {tr_loss:.3f} | test {te_loss:.3f}/{te_acc:.3f} | {time.time()-t0:.1f}s")

train()
torch.save(model.state_dict(), "vit_cifar10.pt")
print("saved -> vit_cifar10.pt")

### 11.2 訓練曲線 / curves

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12,4))
a1.plot(history["train_loss"], label="train loss"); a1.plot(history["test_loss"], label="test loss")
a1.set_title("Loss"); a1.set_xlabel("epoch"); a1.legend()
a2.plot(history["test_acc"], label="test acc", color="tab:green")
a2.set_title("Test accuracy"); a2.set_xlabel("epoch"); a2.legend()
plt.tight_layout(); plt.show()

### 11.3 範例預測（含信心分數）/ predictions

In [ ]:
model.eval()
imgs, labels = next(iter(test_loader))
with torch.no_grad():
    probs = model(imgs.to(DEVICE)).softmax(1).cpu()
conf, pred = probs.max(1)
fig, axes = plt.subplots(2, 8, figsize=(15,4))
for ax, img, lb, pr, cf in zip(axes.flatten(), imgs, labels, pred, conf):
    ax.imshow(denorm(img)); ax.axis("off")
    ax.set_title(f"{CLASSES[pr]} {cf*100:.0f}%", fontsize=9,
                 color=("green" if pr==lb else "red"))
plt.suptitle("預測（綠=正確, 紅=錯誤）/ predictions"); plt.tight_layout(); plt.show()

## 12. 解析 Vision Transformer（重現論文 Fig 6 & 7）
以下分析窺探 ViT 內部表徵。**注意**：部分圖（位置編碼相似度、注意力距離）需要**充分訓練**的模型才會出現清楚的結構；若只跑了少量 epoch，看起來會比較雜訊。

### 12.1 Attention Rollout（Fig. 6）：`[CLS]` 注意到哪裡
單層注意力不足以反映資訊如何跨層流動。**Attention rollout**（Abnar & Zuidema, 2020）把每層注意力（含殘差的恆等項）逐層相乘，近似從輸入到輸出的資訊流。我們取 `[CLS]` 對各 patch 的 rollout 權重，reshape 回 8×8 疊在原圖上——這正是 **DINO 自監督注意力圖**的前身。

In [ ]:
def get_attentions(model, x):
    model.eval(); model.set_store_attn(True)
    with torch.no_grad(): _ = model(x)
    attns = [blk.attn.last_attn for blk in model.blocks]   # 每層 (B, heads, N, N)
    model.set_store_attn(False)
    return attns

def attention_rollout(attns):
    # 逐層 (A + I) 後做 row-normalize，再相乘
    result = None
    for a in attns:
        a = a.mean(1)                                   # 平均所有 head -> (B, N, N)
        a = a + torch.eye(a.size(-1), device=a.device)
        a = a / a.sum(-1, keepdim=True)
        result = a if result is None else a @ result
    return result

imgs, labels = next(iter(test_loader))
attns = get_attentions(model, imgs[:8].to(DEVICE))
roll  = attention_rollout(attns)
cls_to_patch = roll[:, 0, 1:]                            # CLS 對各 patch (8, 64)
maps = cls_to_patch.reshape(-1, cfg.grid, cfg.grid).cpu()
fig, axes = plt.subplots(2, 8, figsize=(15,4))
for i in range(8):
    axes[0,i].imshow(denorm(imgs[i])); axes[0,i].axis("off")
    m = maps[i]; m = (m - m.min())/(m.max()-m.min()+1e-8)
    m = F.interpolate(m[None,None], size=32, mode="bilinear", align_corners=False)[0,0]
    axes[1,i].imshow(denorm(imgs[i])); axes[1,i].imshow(m, cmap="jet", alpha=0.5); axes[1,i].axis("off")
axes[0,0].set_ylabel("input");
plt.suptitle("Attention rollout：[CLS] 的注意力（下排疊圖）/ paper Fig.6"); plt.tight_layout(); plt.show()

### 12.2 線性嵌入濾波器（Fig. 7 left）
Patch embedding 的卷積權重形狀是 `(D, 3, 4, 4)`——可視為 $D$ 個 RGB 濾波器。對它做 PCA，前幾個主成分常呈現類似 Gabor / 顏色對比的基底函數，說明第一層學到 patch 內細結構的低維表示。

In [ ]:
from sklearn.decomposition import PCA
W = model.patch_embed.proj.weight.detach().cpu()         # (D, 3, P, P)
D = W.shape[0]; flat = W.reshape(D, -1).numpy()
k = min(28, D, flat.shape[1])
comps = PCA(n_components=k).fit(flat).components_.reshape(k, 3, cfg.patch_size, cfg.patch_size)
fig, axes = plt.subplots(4, 7, figsize=(8,5))
for i, ax in enumerate(axes.flatten()):
    if i < k:
        c = comps[i].transpose(1,2,0); c = (c-c.min())/(c.max()-c.min()+1e-8)
        ax.imshow(c)
    ax.axis("off")
plt.suptitle("線性嵌入濾波器前 28 個主成分 / RGB embedding filters (Fig.7 left)"); plt.tight_layout(); plt.show()

### 12.3 位置編碼相似度（Fig. 7 center）
每個 patch 位置有一個學到的 `pos_embed` 向量。計算「某位置 vs 所有位置」的餘弦相似度並 reshape 回 8×8：**充分訓練後**，越接近的位置相似度越高，且呈現行/列結構——代表模型**自己從 1D 編碼學出了 2D 影像拓樸**，這解釋了為何更複雜的 2D 編碼幫助不大。

In [ ]:
pos = model.pos_embed.detach().cpu()[0, 1:]              # (N, D) 去掉 CLS
pos = F.normalize(pos, dim=1)
G = cfg.grid
sim = (pos @ pos.t()).reshape(G, G, G, G)
fig, axes = plt.subplots(G, G, figsize=(7,7))
for i in range(G):
    for j in range(G):
        axes[i,j].imshow(sim[i,j], cmap="viridis", vmin=-1, vmax=1); axes[i,j].axis("off")
plt.suptitle("位置編碼餘弦相似度（每格 = 該位置對所有位置）/ Fig.7 center"); plt.tight_layout(); plt.show()

### 12.4 平均注意力距離 vs 深度（Fig. 7 right）
對每個 head，計算「注意力加權的 patch 間像素距離」平均——類似 CNN 的感受野大小。論文發現：**淺層**就有些 head 看很遠（全域），這是 CNN 做不到的；隨深度增加，注意力距離整體變大。

In [ ]:
coords = np.stack(np.meshgrid(np.arange(G), np.arange(G), indexing="ij"), -1).reshape(-1,2) * cfg.patch_size
dist_t = torch.tensor(np.linalg.norm(coords[:,None,:]-coords[None,:,:], axis=-1), dtype=torch.float32)
imgs, _ = next(iter(test_loader))
attns = get_attentions(model, imgs[:64].to(DEVICE))
mean_dist = np.zeros((cfg.depth, cfg.heads))
for l, a in enumerate(attns):
    a = a[:, :, 1:, 1:].mean(0).cpu()        # 去掉 CLS 行列, 對 batch 平均 -> (heads, N, N)
    mean_dist[l] = (a * dist_t).sum(-1).mean(-1).numpy()
plt.figure(figsize=(7,5))
for h in range(cfg.heads):
    plt.scatter(np.arange(1, cfg.depth+1), mean_dist[:,h], label=f"head {h}")
plt.xlabel("網路深度 (layer)"); plt.ylabel("平均注意力距離 (pixels)")
plt.title("Attention distance vs depth / Fig.7 right"); plt.legend(); plt.show()

## 13. 與 `vit-pytorch` 對照 / sanity check
拿社群常用的 `SimpleViT` 對照我們的實作。注意 **`SimpleViT` 的設計選擇與論文/我們不同**：它用**全域平均池化 (GAP)** 取代 `[CLS]`、用**固定 sin-cos 位置編碼**取代可學習編碼、且無 dropout。GAP 版在小資料上常更好訓練，但**沒有 `[CLS]` 注意力可看**——這正是 DINO 保留 `[CLS]` 的原因。

In [ ]:
from vit_pytorch import SimpleViT
sv = SimpleViT(image_size=32, patch_size=4, num_classes=10, dim=192, depth=6, heads=3, mlp_dim=768)
sv_params = sum(p.numel() for p in sv.parameters())
print("SimpleViT 輸出:", tuple(sv(torch.randn(2,3,32,32)).shape), f"| 參數量 {sv_params/1e6:.2f} M")
print(f"我們的 ViT 參數量: {n_params/1e6:.2f} M")
print("差異：SimpleViT = GAP + 固定 sin-cos 位置編碼 + 無 [CLS] + 無 dropout")

## 14. 總結與下一步 / Summary & next steps
**你手刻了什麼**：PatchEmbedding → `[CLS]`+位置編碼 → Multi-Head Self-Attention → Pre-LN Encoder → `[CLS]` 分類頭，完全對應論文 Eq. 1–4，並重現了 Fig 6 & 7 的分析。

**帶走的核心觀念**
1. **圖像 = patch 序列**：ViT 把視覺問題變成序列建模，幾乎照搬 NLP 的 Transformer。
2. **少歸納偏好 = 資料飢渴**：沒有局部性/平移等變性先驗，從零訓練在小資料上吃虧（本筆記本親身驗證）——這就是大規模（自）監督預訓練的動機。
3. **`[CLS]` 注意力 = 可解釋性入口**：rollout 圖顯示模型關注語意相關區域，是 DINO 注意力圖的前身。

**下一步（另一本筆記本）→ DINO / cellDINO**
- 同一個 ViT backbone，把監督式分類頭換成**自監督**目標（學生/教師網路 + 多視角裁剪 + 中心化/銳化），不需標籤就能學出強表徵。
- DINO 的注意力圖能**無監督分割物件**——直接建立在本筆記本的 `[CLS]`-attention 機制上。
- cellDINO：把同套自監督方法搬到細胞/顯微影像領域。